# Reproducing Gao et al. (2018) from DANDI:000007 and CRCNS:alm-4

**Paper.** Gao Z, Davis C, Thomas AM, Economo MN, Abrego AM, Svoboda K, De Zeeuw CI, Li N (2018).
A cortico-cerebellar loop for motor planning. *Nature* 563, 113-116.
[doi:10.1038/s41586-018-0633-x](https://doi.org/10.1038/s41586-018-0633-x). The claims file
(`claims/gao-2018.yaml`) was written from the version of record (PMC6212318) with its Methods,
Extended Data legends and Supplementary Information; the legends of the main Figures 1 to 4 were
not in the text read.

**Datasets.**
- [DANDI:000007](https://dandiarchive.org/dandiset/000007/0.220126.1903), version
  `0.220126.1903` (doi:10.48324/dandi.000007/0.220126.1903). The paper cites no DANDI version
  (the dandiset was created in 2020, after the paper). This is the only published version; the
  draft has the same 54 assets and size, so the published one is pinned.
- [CRCNS:alm-4](https://crcns.org/data-sets/motor-cortex/alm-4) (doi:10.6080/K0NS0S26),
  unversioned (descriptor version 0.6, 23 October 2018). Files read, with their MD5 sums from
  `checksums.md5`: `ALM_recording_DCN_photoactivation/datafiles.tar.gz`
  (12e9d7af630523ed2511826eeacf6b48), `ALM_recording_DCN_photoinhibition/datafiles.tar.gz`
  (5123dc96df58457f853e2deb5ce479c7), and, for method details, the two `analyses_scripts.zip`
  (601f4b52198614dbd4761a64d2658698 and 4639a8478bc307fb3a5a4b8f2a53fb58).

Both datasets hold the same 54 ALM recording sessions of 13 mice (shown below). The analysis
runs on the CRCNS copy, which holds every trial, the GoodTrials flag and the laser traces; the
DANDI copy is checked against it unit by unit and trial by trial, and the counts of the Fig. 2
claim are also given on the DANDI trial set.

Neither dataset holds the lesion experiments, the behavior-only sessions, the CN recordings,
lick times, video, the VGAT-ChR2-EYFP experiments or the anatomy; the claims on those are
blocked as `data_missing` (table below).

| claim | tested |
|---|---|
| fig1d-fastigial-lesion-contra-choice | blocked, data_missing |
| fig1e-fastigial-lesion-slow-initiation | blocked, data_missing |
| fig1d-dentate-lesion-no-effect | blocked, data_missing |
| fig1g-fastigial-activation-early-sample-no-effect | tested (recording sessions) |
| fig1g-fastigial-activation-delay-errors | tested (recording sessions) |
| fig1h-fastigial-activation-delay-bias | tested, reading (a) only |
| fig1h-fastigial-activation-response-contra-bias | blocked, data_missing |
| fig1i-fastigial-activation-licking-unaffected | blocked, data_missing; early-lick part tested |
| fig1g-dentate-activation-little-effect | tested, reading (a), 3 of 8 mice |
| fig1j-photoinhibition-silences-fastigial | blocked, data_missing |
| fig1k-inhibition-early-sample-no-effect | blocked, data_missing |
| fig1k-inhibition-late-sample-delay-errors | blocked, data_missing |
| fig1k-inhibition-delay-bias | tested (recording mice) |
| fig1k-inhibition-response-ipsi-bias | blocked, data_missing |
| fig1l-inhibition-execution-unaffected | blocked, data_missing; early-lick part tested |
| ed10-vgat-fastigial-perturbation-performance | blocked, data_missing |
| fig2-alm-selective-fraction | tested |
| fig2-cn-selective-fraction | blocked, data_missing |
| fig2-preparatory-more-in-alm | tested, ALM only |
| fig2-peri-movement-equal | tested, ALM only |
| ed4-lick-modulated-few | blocked, data_missing |
| fig2cd-selectivity-emerges-gradually | tested, ALM only |
| ed4e-cn-predicts-movement | blocked, data_missing |
| ed4f-preparatory-all-nuclei | blocked, data_missing |
| ed4g-fastigial-contra-late-delay | blocked, data_missing |
| fig3-cn-altered-by-alm-inhibition | blocked, data_missing |
| fig3b-cn-net-disinhibition | blocked, data_missing |
| fig3c-cn-selectivity-abolished | blocked, data_missing |
| ed6-cn-selectivity-both-alm-hemispheres | blocked, data_missing |
| fig3d-alm-altered-by-cn-inhibition | tested |
| fig3e-alm-mean-rate-unchanged | tested |
| fig3e-alm-increase-decrease | tested |
| fig3f-alm-selectivity-abolished-by-cn-inhibition | tested |
| fig4bc-fastigial-targets-alm-thalamus | blocked, data_missing |
| ed8-snr-sc-converge | blocked, data_missing |
| fig4-fastigial-activation-alm-latency | tested |
| fig4e-fastigial-activation-abolishes-alm-selectivity | tested |
| fig4f-fastigial-effect-persists | tested |
| fig4g-dentate-effect-recovers | tested |
| fig4i-cd-control-separation | tested |
| fig4i-fastigial-collapses-cd | tested |
| cd-captures-selectivity | tested |
| ed10b-fastigial-activation-errors-during-recording | tested |
| ed10c-alm-inhibition-endpoints-predict | blocked, data_missing |
| ed10d-fastigial-endpoints-no-prediction | tested |
| ed10ef-alm-drives-licking-after-fastigial | blocked, data_missing |

Method details the claims file leaves open are taken from the paper's Methods and, where those
are silent, from the analysis scripts deposited in CRCNS:alm-4 (the paper's reporting summary
says its analysis scripts would be deposited on CRCNS.org). Each is cited in `gao2018.py` or in
the claim's section.

In [ ]:
import time
import resource
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

import gao2018 as G

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

started = time.time()
dandi = G.load_dandi()
print(f"DANDI:{G.DANDISET_ID} version {G.DANDISET_VERSION}: {len(dandi)} NWB files streamed "
      f"in {time.time() - started:.0f} s")
crcns, meta = {}, {}
for which in G.CRCNS_FILES:
    s_, m_ = G.stream_crcns(which)
    crcns.update(s_)
    meta.update(m_)
print(f"CRCNS:{G.CRCNS_ID} (doi:{G.CRCNS_DOI}, unversioned): {len(crcns)} sessions, "
      f"{time.time() - started:.0f} s so far, peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")
sessions = G.build_sessions(crcns, meta)

## Data check

In [ ]:
info = pd.DataFrame([dict(session=s.key, mouse=s.mouse, group=s.group, cn=s.cn, cn_side=s.cn_side,
                          alm_ml=s.alm_ml, trials=len(s.trials),
                          good=int(s.trials.good.sum()),
                          units=len(s.units), pyramidal=len(s.pyramidal()),
                          FS=sum(u['cell_type'] == 'FS' for u in s.units),
                          unclassified=sum(u['cell_type'] not in ('pyramidal', 'FS') for u in s.units))
                     for s in sessions])
print("Sessions and units by group and mouse:")
print(info.groupby(["group", "mouse"]).agg(sessions=("session", "size"), units=("units", "sum"),
                                         pyramidal=("pyramidal", "sum"), FS=("FS", "sum"),
                                         unclassified=("unclassified", "sum"),
                                         cn=("cn", "first"), cn_side=("cn_side", "first"),
                                         alm_ml=("alm_ml", "first")).to_string())
print("\nTotals by group:")
print(info.groupby("group")[["units", "pyramidal", "FS", "unclassified"]].sum().to_string())
print("\nPhotostimulation codes on good trials, by group:")
ct = pd.concat([s.trials.assign(group=s.group) for s in sessions], ignore_index=True)
print(pd.crosstab(ct[ct.good].group, ct[ct.good].ptype.fillna(-1).astype(int)).to_string())
print("\nTrial outcomes on good trials:", ct[ct.good].outcome.value_counts().to_dict())
print("Photostimulation code descriptions in the files:")
for key in ["ANM314746_20151020", "BAYLORCD8_20171018"]:
    print(" ", key, crcns[key]["trial_descr"][4:])
print("\nChannels of the CRCNS time series (timeSeriesArrayHash), sessions per channel set:",
      pd.Series([", ".join(crcns[s.key]["ts_channels"]) for s in sessions]).value_counts().to_dict())

The CRCNS descriptor lists 34 sessions (9 mice) of ALM recording during fastigial or dentate
photo-activation and 20 sessions (4 mice) during CN photo-inhibition; the files hold exactly
these. The fastigial mice are 6 (ANM314746, ANM314757, ANM314758, ANM314760, BAYLORCD6,
BAYLORNL15) and the dentate mice 3 (ANM314756, BAYLORNL12, BAYLORNL14), as Supplemental Table 1
gives (6 and 3). The paper reports 1309 ALM units (1194 putative pyramidal) in 10 mice over 39
sessions for Fig. 2, of which 912 pyramidal neurons in the 9 photo-activation mice, and 490
units (454 pyramidal) in 4 mice over 19 sessions for CN photo-inhibition. The counts above
differ from both (printed): the files hold more units than the paper reports for the same
mice.

In [ ]:
cmp_ = G.compare_deposits(sessions, dandi, crcns)
print(cmp_.to_string(index=False))
print("\nAll units have identical spike times in both deposits:",
      bool((cmp_.units_same_spikes == cmp_.units_crcns).all() and (cmp_.units_dandi == cmp_.units_crcns).all()))
print("Units whose cell type differs (DANDI 'unknown' where CRCNS has none counts as same):",
      int((cmp_.units_crcns - cmp_.cell_type_same).sum()))
print("DANDI trials = CRCNS good trials with a photostimulation code, except in sessions:")
print(cmp_[cmp_.eligible_not_in_dandi > 0][["session", "good_with_code", "dandi_trials", "eligible_not_in_dandi"]].to_string(index=False))
print("DANDI trial codes and outcomes equal CRCNS for every matched trial:",
      bool(cmp_.codes_same.all() and cmp_.outcomes_same.all()))
ex = dandi["BAYLORCD8_20171018"]
print("\nDANDI trial column descriptions:", {k: v for k, v in ex["trial_columns"].items() if k in ("cue_time", "pole_in_time")})
print("DANDI cue_time minus start_time, first trials (s):", (ex["trials"].cue_time - ex["trials"].start_time).head(3).round(3).tolist())
two = [s for s in sessions if s.trials.cue_col2.notna().any()]
d2 = np.concatenate([(s.trials.cue_col2 - s.trials.cue)[s.trials.good].to_numpy() for s in two])
print(f"CRCNS CueTime has two columns in {len(two)} sessions; on good trials the second minus the first: "
      f"median {np.nanmedian(d2):.3f} s, nonzero in {int(np.nansum(np.abs(d2) > 1e-6))} of {int(np.isfinite(d2).sum())} trials "
      "(the scripts use the first column)")
print("\nTrials eligible in CRCNS (good, with a code) but missing from DANDI, by CRCNS trial number (1-based):")
for s in sessions:
    elig = (s.trials.good & s.trials.ptype.notna()).to_numpy()
    miss = np.flatnonzero(elig & ~s.trials.in_dandi.to_numpy()) + 1
    if len(miss):
        span = elig[miss.min() - 1:miss.max()]
        print(f"  {s.key}: {len(miss)} trials, numbers {miss.min()} to {miss.max()}; every eligible trial in that range missing: "
              f"{bool(span.sum() == len(miss))}; last trial held by DANDI {np.flatnonzero(s.trials.in_dandi.to_numpy()).max() + 1}; "
              f"trials in the CRCNS table {len(s.trials)}")
print("\nPhotostimulation site and recording location in DANDI (general/optogenetics and electrode group locations), "
      "one session per photo-inhibition mouse, with the CRCNS values:")
for m in ["BAYLORCD8", "BAYLORCD12", "BAYLORCD13", "BAYLORCD14"]:
    s = next(s for s in sessions if s.mouse == m)
    print(f"  {s.key}: DANDI optogenetics {dandi[s.key]['ogen']}; DANDI electrode groups {dandi[s.key]['egroups']}; "
          f"CRCNS CN side {s.cn_side}, CRCNS ALM ML {s.alm_ml}")

The two deposits hold the same recordings: every unit's spike times are identical, and every
DANDI trial matches a CRCNS trial with the same photostimulation code and outcome. Each
recording is counted once here. DANDI keeps only the trials with GoodTrials = 1 and a valid
code, which the scripts also use, but in two sessions it lacks trials CRCNS holds (printed
above). DANDI rounds trial start times (differences up to 0.05 s) and describes `cue_time`
and `pole_in_time` as relative to the trial start, while the values are in session time.

In [ ]:
# Units against the lab's compiled lists in analyses_scripts.zip, read only to reconcile counts
lab_fig2 = G.read_script_mat("photoactivation", "analysis_compile_all_ALM_data.mat", ["cellType_all", "sig_selective", "file_list"])
lab_fr = G.read_script_mat("photoactivation", "analysis_compiled_all_ALM_DCNstim_FR.mat", ["cellType_all", "p_stim2_FN", "p_stim2_DN", "file_list"])
lab_inh = G.read_script_mat("photoinhibition", "analysis_compiled_all_ALM_DCNsilencing_selectivity.mat", ["cellType_all", "file_list"])
lab_cd = G.read_script_mat("photoactivation", "analysis_compiled_all_ALM_DCNstim_CD.mat", ["session_name", "neuronlist_allSession"])
# the scripts read the files in name order and the units in file order
pa_units = [(s, u) for s in sessions if s.group != "inhibition" for u in s.units]
in_units = [(s, u) for s in sessions if s.group == "inhibition" for u in s.units]
pa_row = {(s.key, u["key"]): j for j, (s, u) in enumerate(pa_units)}
for nm, lab, keys in [("photo-activation", lab_fr, [s.key for s in sessions if s.group != "inhibition"]),
                      ("photo-inhibition", lab_inh, [s.key for s in sessions if s.group == "inhibition"])]:
    files = [str(x["name"]) for x in np.atleast_1d(lab["file_list"])]
    print(f"{nm} compile: {len(files)} files, in the same order as the sessions here: {files == [f'data_structure_{k}.mat' for k in keys]}")
ct_code = {"pyramidal": 1, "FS": 2}
for nm, lab, units in [("Fig. 2 compile (photo-activation)", lab_fig2, pa_units), ("FR compile (photo-activation)", lab_fr, pa_units),
                       ("selectivity compile (photo-inhibition)", lab_inh, in_units)]:
    ct_lab = np.asarray(lab["cellType_all"]).astype(int)
    ct_here = np.array([ct_code.get(u["cell_type"], 0) for s, u in units])
    print(f"{nm}: units {len(ct_lab)} (here {len(units)}), pyramidal {int((ct_lab == 1).sum())} (here {int((ct_here == 1).sum())}), "
          f"same cell type for every unit: {bool(len(ct_lab) == len(ct_here) and (ct_lab == ct_here).all())}")
pyr_lab = np.asarray(lab_fig2["cellType_all"]) == 1
print("Fig. 2 compile: pyramidal units marked selective (sig_selective) in any epoch:",
      int((pyr_lab & (np.nan_to_num(lab_fig2["sig_selective"]) > 0).any(1)).sum()), "of", int(pyr_lab.sum()))
pyr_fr = np.asarray(lab_fr["cellType_all"]) == 1
print("FR compile: pyramidal units with photostimulation p-values (tested), fastigial delay",
      int((pyr_fr & ~np.isnan(lab_fr["p_stim2_FN"]).all(1)).sum()), ", dentate delay", int((pyr_fr & ~np.isnan(lab_fr["p_stim2_DN"]).all(1)).sum()))
lab_cd_sessions = [str(x) for x in np.atleast_1d(lab_cd["session_name"])]
print(f"CD compile: {len(lab_cd_sessions)} sessions, {min(np.size(x) for x in lab_cd['neuronlist_allSession'])} to "
      f"{max(np.size(x) for x in lab_cd['neuronlist_allSession'])} neurons:", lab_cd_sessions)

# Spike width (trough to peak of the mean waveform) against the files' cellType label; E1 puts
# pyramidal neurons above 0.45 ms and FS below 0.35 ms. The Methods give 25 kHz (NeuroNexus probes)
# and 20 kHz (Intan) sampling; both are shown for each year.
wd = []
for s in sessions:
    for u in s.units:
        w = u["waveform"]; i0 = int(np.argmin(w)); j0 = i0 + int(np.argmax(w[i0:]))
        wd.append(dict(year=s.date[:4], label=u["cell_type"] or "none", samples=len(w), tp=j0 - i0))
wd = pd.DataFrame(wd)
print("\nSamples per mean waveform by year:", wd.groupby("year").samples.agg(lambda x: sorted(set(x))).to_dict())
for fs in (25000, 20000):
    width = wd.tp / fs * 1000
    cls = np.where(width > 0.45, "> 0.45 ms", np.where(width < 0.35, "< 0.35 ms", "0.35 to 0.45 ms"))
    print(f"spike width at {fs} Hz against the label:")
    print(pd.crosstab([wd.year, wd.label], cls).to_string())
w20 = wd.tp / 20000 * 1000
odd = wd[(wd.label == "pyramidal") & (wd.year != "2015") & (w20 <= 0.45 + 1e-9)]
print("2017 and 2018 units labelled pyramidal with width <= 0.45 ms at 20 kHz, by width (ms):",
      (odd.tp / 20000 * 1000).round(3).value_counts().sort_index().to_dict())

The lab's compiled analysis files hold the same units as the deposits: 1186 units (1084
pyramidal) in the 34 photo-activation sessions and 737 (682 pyramidal) in the 20 photo-inhibition
sessions, with the same cell type for every unit (printed). The difference from the paper's counts
(912 pyramidal neurons in the photo-activation mice, 454 of 490 units in 19 photo-inhibition
sessions) is therefore not a selection that these files record. The lab's coding-direction compile
lists its sessions (printed); they are compared with the sessions used here in the coding direction
section. Spike widths agree with the labels at 25 kHz for the 2015 sessions and at 20 kHz for the
2017 and 2018 sessions, with the exceptions printed.

In [ ]:
lat = []
for s in sessions:
    t = s.trials[s.trials.good & s.trials.ptype.notna()]
    for c in sorted(t.ptype.unique()):
        tc = t[t.ptype == c]
        ref = np.where(np.isin(c, [1, 3]), tc.pole_in, tc.pole_out) if c > 0 else tc.pole_out
        on = tc.laser_on - ref
        lat.append(dict(session=s.key, form=crcns[s.key]["laser_form"], code=int(c), n=len(tc),
                        lit=round(float(tc.laser_on.notna().mean()), 2),
                        at_named_epoch=round(float((np.abs(on) < 0.02).mean()), 2),
                        onset_median_ms=round(float(np.nanmedian(on)) * 1000, 1) if on.notna().any() else np.nan,
                        onset_iqr_ms=round(float(np.nanpercentile(on, 75) - np.nanpercentile(on, 25)) * 1000, 1) if on.notna().any() else np.nan,
                        dur_median_s=round(float(np.nanmedian(tc.laser_off - tc.laser_on)), 3) if on.notna().any() else np.nan,
                        mean_power=round(float(np.nanmedian(tc.laser_mean[tc.laser_on.notna()])), 2) if on.notna().any() else np.nan,
                        peak=round(float(np.nanmedian(tc.laser_peak)), 2)))
lat = pd.DataFrame(lat)
print("Laser power trace ('laser_power' channel) per session and code: fraction of trials with light,")
print("onset relative to the epoch the code names (sample: pole in; delay: pole out), duration, mean power while on, peak:")
print(lat[lat.code > 0].to_string(index=False))
print("\nControl trials with light:", int((lat[lat.code == 0].lit > 0).sum()), "sessions of", int((lat.code == 0).sum()))
bad = lat[lat.code.isin([1, 2, 3, 4]) & ((lat.dur_median_s < 0.40) | (lat.dur_median_s > 0.55) | (lat.at_named_epoch < 0.75))]
MISMATCH = sorted(bad.session.unique())
print("Photo-activation sessions whose light disagrees with the codes (median duration outside 0.40 to 0.55 s, "
      "or fewer than 75% of trials lit at the named epoch):", MISMATCH)

Laser traces: in the 2017 and 2018 sessions (stored one row per trial) the light starts within
2 ms of the epoch the code names, lasts 0.455 s for photo-activation (10 pulses at 20 Hz) and
0.494 s or 1.295 s for codes 5 and 6, as the descriptor and Methods say, and no control trial is
lit. The 2015 sessions store the series as one long series with a trial number per sample; their
light lasts 0.500 s from the named epoch, except in the sessions listed as `MISMATCH` above. In
ANM314746_20151020 codes 1 and 2 (described as 500 ms) have 1.3 s of light; in
ANM314756_20151020 the light lasts a median 0.83 and 0.78 s; ANM314757_20151021 shows almost no
light (peak 0.17 mW) and its onsets do not match the trials; in BAYLORNL14_20170603 only 54% of
code 3 and 39% of code 4 trials are lit at the named epoch, the others at the other epoch (onset
IQR about 1300 ms). The main analyses keep these sessions, since the codes are what the lab's
scripts use and its coding-direction compile includes three of them; every claim that uses codes
1 to 4 also shows its result without them. The power values are those of the `laser_power`
channel, which the files say is in mW.

In [ ]:
# Clock check: trial-type selectivity should be absent before the sample epoch and appear after it,
# and photostimulation should change ALM rates within milliseconds of the light onset.
win = np.arange(-1.0, 1.21, 0.2)    # window starts relative to pole in (sample onset), 200 ms windows
frac = []
for grp_name, inh_group in [("photo-activation", False), ("photo-inhibition", True)]:
    for shift_name, shift in [("true", 0.0), ("events +0.6 s", 0.6)]:
        nsig = np.zeros(len(win)); ntot = 0
        for s in sessions:
            if (s.group == "inhibition") != inh_group:
                continue
            t = s.trials
            for u in s.pyramidal():
                m = s.base_mask() & (t.ptype == 0).to_numpy() & t.correct.to_numpy() & s.stable(u, len(t))
                R, L = m & (t.instructed == "R").to_numpy(), m & (t.instructed == "L").to_numpy()
                if R.sum() < 5 or L.sum() < 5:
                    continue
                ntot += 1
                pi = t.pole_in.to_numpy() + shift
                for k, w0 in enumerate(win):
                    c = s.counts(u, pi + w0, pi + w0 + 0.2)
                    if np.var(c[R]) + np.var(c[L]) > 0 and stats.ttest_ind(c[R], c[L]).pvalue < 0.05:
                        nsig[k] += 1
        frac.append(pd.Series(nsig / ntot, index=np.round(win, 1), name=f"{grp_name}, {shift_name} (n={ntot})"))
print("Fraction of pyramidal neurons selective (t-test p<0.05) in 200 ms windows starting at the given time")
print("from pole in (sample onset), with true and shifted alignment, by session group:")
print(pd.concat(frac, axis=1).round(3).to_string())

In both session groups, with the true alignment the fraction of selective neurons is near the
5% false-positive rate before sample onset and rises in the first window after it; with the trial
events taken 0.6 s later the rise moves 0.6 s earlier relative to the shifted events (printed).
Spikes and trial events share a clock in the photo-activation and the photo-inhibition sessions to
within the 200 ms of a window. The photostimulation responses in the claim sections below (latencies
of a few ms from the light onset for photo-activation) bound the offset more tightly for those
trials.

## Behavior in the recording sessions

The claims on Fig. 1g, 1h and 1k concern behavior sessions; the datasets hold only the
recording sessions, in which photostimulation was on 33% of trials (Methods). Supplemental
Table 1 shows that the 6 fastigial photo-activation mice of Fig. 1g are the 6 recorded ones,
so the recording sessions hold part of their behavior. Performance is the fraction correct
excluding early-lick and no-lick trials, on good trials (B1). Contra and ipsi are relative to
the manipulated CN, which the Methods and the photostimulation codes put contralateral to the
recorded ALM; the files' CN side disagrees for three photo-inhibition mice (see the summary),
so both are run where it matters.

In [ ]:
fa = [s for s in sessions if s.group == "fastigial"]
de = [s for s in sessions if s.group == "dentate"]
inh = [s for s in sessions if s.group == "inhibition"]
tab_fa = G.perf_table(fa, [0, 1, 2])
tab_de = G.perf_table(de, [0, 3, 4])
tab_in = G.perf_table(inh, [0, 5, 6])
tab_in_meta = G.perf_table(inh, [0, 5, 6], contra_from="metadata")


def show(r):
    return {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in r.items()}

### fig1g-fastigial-activation-early-sample-no-effect

> Fastigial photo-activation during the early sample epoch does not affect behavioral choice
> for either trial type.

Criterion: per trial type, control against early sample photo-activation (code 1, 0.455 s),
one-sided bootstrap of B1; reproduced when p > 0.05 for both trial types.

In [ ]:
rng = np.random.default_rng(11)
for side in ["contra", "ipsi"]:
    print(side, show(G.hier_boot(tab_fa, 1, side, rng=rng)))
print("pooled", show(G.hier_boot(tab_fa, 1, None, rng=rng)))
print("\nPer mouse (pooled types), control and sample photo-activation:")
for m, g in tab_fa.groupby("mouse"):
    r = G.hier_boot(g, 1, None, rng=rng, n_boot=2000)
    print(f"  {m}: control {r['ctrl']:.3f} (n={r['n_ctrl']}), stim {r['stim']:.3f} (n={r['n_stim']})")
print("\nWithout the sessions in MISMATCH:")
for side in ["contra", "ipsi"]:
    print(" ", side, show(G.hier_boot(tab_fa[~tab_fa.session.isin(MISMATCH)], 1, side, rng=rng)))
print("Trial-level checks (not the criterion's test), all sessions:", show(G.trial_level_tests(tab_fa, 1)))

**Result: reproduced.** Early sample photo-activation does not lower performance on either trial
type: contralateral 0.857 to 0.826 (one-sided p for a drop 0.36), ipsilateral 0.887 to 0.899
(p = 0.64). Without the sessions in MISMATCH, p = 0.29 and 0.81; the trial-level checks, which are
less conservative than the bootstrap, give p = 0.28 (z-test) and 0.35 (stratified by session).

### fig1g-fastigial-activation-delay-errors

> Fastigial photo-activation during the early delay epoch lowers behavioral performance, even
> though the photostimulus ends 800 ms before the movement.

Criterion: pooled performance, control against early delay photo-activation (code 2), one-sided
bootstrap p < 0.05. By the laser traces above the light lasts 0.455 s from delay onset in the
2017 sessions, ending 0.845 s before the go cue, and 0.500 s in the 2015 sessions, ending 0.8 s
before it, except in ANM314746_20151020, where it lasts 1.3 s, up to the go cue (one of the
sessions in MISMATCH).

In [ ]:
rng = np.random.default_rng(12)
r = G.hier_boot(tab_fa, 2, None, rng=rng)
print(show(r))
print("p reported as the fraction of 10,000 resamples without a drop; 0 means < 0.0001")
for m, g in tab_fa.groupby("mouse"):
    rr = G.hier_boot(g, 2, None, rng=rng, n_boot=2000)
    print(f"  {m}: control {rr['ctrl']:.3f}, stim {rr['stim']:.3f}, change {rr['change']:+.3f}")
print("without the sessions in MISMATCH:", show(G.hier_boot(tab_fa[~tab_fa.session.isin(MISMATCH)], 2, None, rng=rng)))

**Result: reproduced.** Early delay photo-activation lowers pooled performance from 0.870 to 0.630
(no resample of 10,000 without a drop, p < 0.0001), and every mouse drops, by 0.083 to 0.450.
Without the sessions in MISMATCH, 0.871 to 0.619 (p = 0.0004).

### ed10b-fastigial-activation-errors-during-recording

> In the ALM recording sessions, contralateral fastigial photo-activation lowers behavioral
> performance, with "lick left" and "lick right" trials pooled, in the 6 mice.

Criterion: same comparison as above, reproduced when the one-sided hierarchical bootstrap gives
p < 0.001. The numbers are those printed for fig1g-fastigial-activation-delay-errors; they are
repeated here with their own generator.

In [ ]:
rng = np.random.default_rng(13)
r = G.hier_boot(tab_fa, 2, None, rng=rng)
print(show(r))
print("mice:", r["n_mice"], "sessions:", r["n_sessions"])
print("without the sessions in MISMATCH:", show(G.hier_boot(tab_fa[~tab_fa.session.isin(MISMATCH)], 2, None, rng=rng)))

**Result: reproduced.** In the 6 mice (21 sessions) performance falls from 0.870 to 0.630 with
p < 0.0001, below 0.001; without the sessions in MISMATCH (19 sessions) also p < 0.0001.

### fig1h-fastigial-activation-delay-bias

> Fastigial photo-activation during the delay epoch biases future movements toward either the
> contralateral or the ipsilateral direction, depending on the mouse.

Criterion: per mouse, change in performance on contralateral and ipsilateral trials, per-mouse
one-sided bootstrap (sessions and trials of that mouse) at p < 0.05; reproduced when under one
reading at least one mouse has a significant contralateral drop larger than its ipsilateral drop
and at least one a significant ipsilateral drop larger than its contralateral drop. Reading (a),
early delay 0.455 s (code 2), is in the data; reading (b), the 1.255 s whole-delay experiment,
is not.

In [ ]:
rng = np.random.default_rng(14)
def bias_table(tab, code_, rng):
    rows = []
    for m, g in tab.groupby("mouse"):
        rc = G.hier_boot(g, code_, "contra", rng=rng); ri = G.hier_boot(g, code_, "ipsi", rng=rng)
        rows.append(dict(mouse=m, contra_ctrl=rc["ctrl"], contra_stim=rc["stim"], contra_change=rc["change"], contra_p=rc["p_drop"],
                         ipsi_ctrl=ri["ctrl"], ipsi_stim=ri["stim"], ipsi_change=ri["change"], ipsi_p=ri["p_drop"]))
    b = pd.DataFrame(rows)
    b["contra_biased"] = (b.contra_p < 0.05) & (b.contra_change < b.ipsi_change)
    b["ipsi_biased"] = (b.ipsi_p < 0.05) & (b.ipsi_change < b.contra_change)
    return b


b = bias_table(tab_fa, 2, rng)
print(b.round(4).to_string(index=False))
print("mice with a significant larger contralateral drop:", int(b.contra_biased.sum()),
      "; with a significant larger ipsilateral drop:", int(b.ipsi_biased.sum()))
b2 = bias_table(tab_fa[~tab_fa.session.isin(MISMATCH)], 2, rng)
print("\nWithout the sessions in MISMATCH:")
print(b2[["mouse", "contra_change", "contra_p", "ipsi_change", "ipsi_p", "contra_biased", "ipsi_biased"]].round(4).to_string(index=False))

**Result: reproduced** under reading (a). ANM314760 and BAYLORNL15 drop on contralateral trials (by
0.726 and 0.719, p < 0.0001) and ANM314746 and ANM314757 on ipsilateral trials (by 0.391, p = 0.003,
and 0.221, p = 0.029), each more than on the other side. The same four mice are biased without the
sessions in MISMATCH (ipsilateral p = 0.018 and 0.005). Reading (b) is not in the data.

### fig1g-dentate-activation-little-effect

> Dentate photo-activation produces little effect on behavioral choice.

Criterion: pooled performance, control against delay dentate photo-activation, one-sided
bootstrap; a claim of absence, reproduced when it does not find a drop at p < 0.05 (paper
p = 0.06). Reading (a), early delay 0.455 s (code 4), in the 3 recorded dentate mice; the paper's
8 dentate mice and reading (b) are not in the data.

In [ ]:
rng = np.random.default_rng(15)
print("pooled", show(G.hier_boot(tab_de, 4, None, rng=rng)))
for side in ["contra", "ipsi"]:
    print(side, show(G.hier_boot(tab_de, 4, side, rng=rng)))
print("sample epoch (code 3), pooled, for reference:", show(G.hier_boot(tab_de, 3, None, rng=rng)))
tab_de_ok = tab_de[~tab_de.session.isin(MISMATCH)]
print("\nWithout the sessions in MISMATCH, pooled:", show(G.hier_boot(tab_de_ok, 4, None, rng=rng)))
for side in ["contra", "ipsi"]:
    print("  ", side, show(G.hier_boot(tab_de_ok, 4, side, rng=rng)))
print("Trial-level checks (not the criterion's test): all sessions", show(G.trial_level_tests(tab_de, 4)))
print("                                     without MISMATCH", show(G.trial_level_tests(tab_de_ok, 4)))
print("Per mouse (all sessions):")
for m, g in tab_de.groupby("mouse"):
    rr = G.hier_boot(g, 4, None, rng=rng)
    print(f"  {m}: control {rr['ctrl']:.3f}, stim {rr['stim']:.3f}, change {rr['change']:+.3f}, p {rr['p_drop']:.3f}")

**Result: partially reproduced.** On all 13 sessions the bootstrap finds no drop at p < 0.05
(0.873 to 0.835, change -0.039, p = 0.107; paper p = 0.06), as the criterion asks. Without the two
sessions whose light disagrees with code 4 it finds one (0.876 to 0.824, change -0.052, p = 0.042),
and the trial-level checks find a drop on all sessions (z-test p = 0.007, stratified p = 0.016). The
effect is small, as "little effect" says, but its absence at p < 0.05 depends on which sessions are
kept. This is 3 of the paper's 8 dentate mice, under reading (a) only.

### fig1k-inhibition-delay-bias

> Fastigial photo-inhibition during the delay epoch biases future movements to the
> contralateral or the ipsilateral direction, depending on the mouse.

Criterion: as fig1h-fastigial-activation-delay-bias, on delay photo-inhibition trials; readings
(a) 0.5 s early delay (code 5; the late delay is not in the data) and (b) 1.3 s whole delay
(code 6). The data hold the 4 recorded L7-cre x Ai32 mice (CD8, CD12, CD13, CD14), not the 7
and 3 behavior mice of Extended Data Fig. 2h and 2g (of which CD12 and CD13 are in both). Contra
and ipsi from the paper's rule (CN contralateral to the recorded ALM) and, as a variant, from
the files' CN side.

In [ ]:
rng = np.random.default_rng(16)
for lab, tab in [("paper rule", tab_in), ("files' CN side", tab_in_meta)]:
    for code_ in (5, 6):
        rows = []
        for m, g in tab.groupby("mouse"):
            rc = G.hier_boot(g, code_, "contra", rng=rng); ri = G.hier_boot(g, code_, "ipsi", rng=rng)
            rows.append(dict(mouse=m, contra_change=rc["change"], contra_p=rc["p_drop"], ipsi_change=ri["change"], ipsi_p=ri["p_drop"],
                             n_stim=rc["n_stim"] + ri["n_stim"]))
        b = pd.DataFrame(rows)
        b["contra_biased"] = (b.contra_p < 0.05) & (b.contra_change < b.ipsi_change)
        b["ipsi_biased"] = (b.ipsi_p < 0.05) & (b.ipsi_change < b.contra_change)
        print(f"{lab}, code {code_} ({G.CODES[code_]}):")
        print(b.round(4).to_string(index=False))
        print(f"  contra-biased mice {int(b.contra_biased.sum())}, ipsi-biased mice {int(b.ipsi_biased.sum())}\n")
print("Pooled over mice:", {c: show(G.hier_boot(tab_in, c, None, rng=rng)) for c in (5, 6)})

**Result: reproduced** under reading (b), code 6. With the paper's side rule BAYLORCD12 and
BAYLORCD13 drop on contralateral trials (by 0.464, p = 0.0015, and 0.385, p = 0.009) and BAYLORCD14
on ipsilateral trials (by 0.234, p = 0.008); with the files' CN side the labels of these three
swap, giving 1 contra- and 2 ipsi-biased mice. Under reading (a), code 5, 0 and 2 (paper's rule) or
1 and 1 (files' side). These are the 4 recorded mice, not the behavior cohort of Extended Data Fig. 2.

### Early licks (parts of fig1i-fastigial-activation-licking-unaffected and fig1l-inhibition-execution-unaffected)

> fig1i: Fastigial photo-activation does not block or change the frequency and duration of
> licking after a lick bout starts, and photo-activation in the delay does not evoke early licks.
>
> fig1l: Fastigial photo-inhibition minimally affects movement execution, and at the intensities
> of the task evokes no consistent movement.

Both claims stay blocked: their lick-bout measures need lick times, which neither dataset holds
(the CRCNS time series hold only the laser channels, printed in the data check; DANDI holds no
behavioral time series), and response-epoch photostimulation, which the recording sessions lack.
The early-lick part of each criterion can be tested from the per-trial LickEarly flag: early-lick
rate in control against delay photostimulation trials, with the bootstrap of B1 two-sided at
p < 0.05 (the criteria's choice), pooled over mice and per mouse. In each criterion the claim
fails if any of its parts differs, so a difference here would decide it. Fastigial
photo-activation: early delay (code 2), and early sample (code 1) for reference; photo-inhibition:
codes 5 and 6 in the 4 recorded mice (the criterion names 7 behavior mice).

In [ ]:
rng = np.random.default_rng(17)
for nm, ss, codes in [("fastigial photo-activation", fa, [2, 1]), ("CN photo-inhibition", inh, [5, 6])]:
    tab = G.early_lick_table(ss, [0] + codes)
    for c in codes:
        r = G.hier_boot(tab, c, None, rng=rng)
        print(f"{nm}, code {c} ({G.CODES[c]}): early-lick rate control {r['ctrl']:.3f} (n={r['n_ctrl']}), stim {r['stim']:.3f} "
              f"(n={r['n_stim']}), change {r['change']:+.3f}, two-sided p = {min(1, 2 * min(r['p_drop'], 1 - r['p_drop'])):.3f}, "
              f"{r['n_mice']} mice, {r['n_sessions']} sessions")
        for m, g in tab.groupby("mouse"):
            rr = G.hier_boot(g, c, None, rng=rng)
            print(f"   {m}: control {rr['ctrl']:.3f}, stim {rr['stim']:.3f} (n={rr['n_stim']}), "
                  f"two-sided p = {min(1, 2 * min(rr['p_drop'], 1 - rr['p_drop'])):.4f}")
        if nm.startswith("fastigial"):
            r = G.hier_boot(tab[~tab.session.isin(MISMATCH)], c, None, rng=rng)
            print(f"   without MISMATCH: control {r['ctrl']:.3f}, stim {r['stim']:.3f}, two-sided p = {min(1, 2 * min(r['p_drop'], 1 - r['p_drop'])):.3f}")

**Result: both claims stay blocked (data_missing).** Where it can be tested, the early-lick part
holds pooled over mice, not in every mouse: the fraction of trials with an early lick goes from 0.083 to 0.130 with early delay fastigial
photo-activation (two-sided
p = 0.33, 6 mice), CN photo-inhibition code 5 0.109 to 0.164 (p = 0.13) and code 6 0.112 to 0.068
(p = 0.22), 4 mice. Single mice differ: BAYLORCD6 licks early more under fastigial photo-activation
(0.081 to 0.324 in the delay, 0.304 in the sample epoch, p < 0.0001), and BAYLORCD8 more under code 5
(0.083 to 0.209, p = 0.008) and less under code 6 (0.090 to 0.011, p < 0.0001); in these mice
photostimulation does change early licking. The lick-bout parts
need lick times, which are not deposited, so neither claim can be decided.

## ALM selectivity (Fig. 2)

Units as E1 (pyramidal), in the 34 photo-activation sessions (the Fig. 2 population; the paper's
39 sessions include a tenth mouse that is not deposited). Selectivity test E2 on control trials.
Readings: trials (a) correct, (b) correct and error grouped by instructed type; response window
(a) 1.3 s, (b) 1.5 s after the go cue. The deposited Fig. 2 script uses (a) and (a).

In [ ]:
t0 = time.time()
sel_rows = []
for s in sessions:
    if s.group == "inhibition":
        continue
    for u in s.pyramidal():
        for tr in ("correct", "all"):
            for rw in (1.3, 1.5):
                r = G.selectivity_tests(s, u, tr, rw)
                r.update(session=s.key, mouse=s.mouse, unit=u["key"], trials=tr, response_s=rw)
                sel_rows.append(r)
sel = pd.DataFrame(sel_rows)
p = sel[["sample", "delay", "response"]].fillna(1) < 0.05
sel["selective"] = p.any(axis=1)
sel["preparatory"] = p[["sample", "delay"]].any(axis=1)
sel["peri"] = p["response"]
sel["testable"] = sel[["sample", "delay", "response"]].notna().any(axis=1)
summ = sel.groupby(["trials", "response_s"]).agg(neurons=("unit", "size"), testable=("testable", "sum"),
                                                  selective=("selective", "sum"), preparatory=("preparatory", "sum"),
                                                  peri=("peri", "sum"))
summ["frac_selective"] = summ.selective / summ.neurons
summ["prep_of_all"] = summ.preparatory / summ.neurons
summ["prep_of_selective"] = summ.preparatory / summ.selective
summ["peri_of_all"] = summ.peri / summ.neurons
summ["peri_of_selective"] = summ.peri / summ.selective
print(summ.round(3).to_string())
print(f"({time.time() - t0:.0f} s)")

### fig2-alm-selective-fraction

> Most ALM putative pyramidal neurons (980 of 1194) distinguish "lick left" from "lick right"
> trials in their spike counts.

Criterion: fraction within 0.82 ± 0.082 under one reading; the counts are judged only where the
dataset holds all units (it holds 34 of the 39 sessions).

In [ ]:
print(summ[["neurons", "selective", "frac_selective"]].round(3).to_string())
a = sel[(sel.trials == "correct") & (sel.response_s == 1.3)]
print("\nReading (a, a) by mouse:")
print(a.groupby("mouse").selective.agg(["size", "sum", "mean"]).round(3).to_string())
# the same on the DANDI trial set
rows = []
for s in sessions:
    if s.group == "inhibition":
        continue
    keep = s.trials.in_dandi.to_numpy()
    for u in s.pyramidal():
        t = s.trials
        m = s.base_mask() & (t.ptype == 0).to_numpy() & s.stable(u, len(t)) & t.correct.to_numpy() & keep
        R, L = m & (t.instructed == "R").to_numpy(), m & (t.instructed == "L").to_numpy()
        pi, po, cue = t.pole_in.to_numpy(), t.pole_out.to_numpy(), t.cue.to_numpy()
        ps = []
        for a_, b_ in [(pi, po), (po, cue), (cue, cue + 1.3)]:
            r_ = s.counts(u, a_, b_) / (b_ - a_)
            ps.append(stats.ttest_ind(r_[R], r_[L]).pvalue if R.sum() >= 2 and L.sum() >= 2 and np.var(r_[R]) + np.var(r_[L]) > 0 else 1.0)
        rows.append(min(ps) < 0.05)
print(f"\nOn the DANDI trial set (reading a, a): {sum(rows)} of {len(rows)} selective ({np.mean(rows):.3f})")
# E2 on spike counts (E2's wording) in place of rates (the Fig. 2 script), reading (a, a)
cnt_sel = []
for s in sessions:
    if s.group == "inhibition":
        continue
    for u in s.pyramidal():
        r_ = G.selectivity_tests(s, u, "correct", 1.3, measure="count")
        cnt_sel.append(any(r_[e] < 0.05 for e in ("sample", "delay", "response") if not np.isnan(r_[e])))
print(f"E2 on spike counts (reading a, a): {sum(cnt_sel)} of {len(cnt_sel)} selective ({np.mean(cnt_sel):.3f})")
# the same units in the lab's Fig. 2 compile (sig_selective), unit by unit
lab_sel = (np.nan_to_num(lab_fig2["sig_selective"]) > 0).any(1)
here = a.set_index(["session", "unit"]).selective
same = [bool(here.loc[(s.key, u["key"])]) == bool(lab_sel[pa_row[(s.key, u["key"])]]) for s in sessions if s.group != "inhibition" for u in s.pyramidal()]
print(f"Reading (a, a) against the lab's compiled sig_selective: same verdict for {sum(same)} of {len(same)} pyramidal units")

**Result: reproduced.** 856 of 1084 pyramidal neurons (0.790) are selective with correct trials and
a 1.3 s response window; the four readings give 0.740 to 0.797, all within 0.82 +- 0.082. Spike
counts in place of rates give the same 856, and the verdict matches the lab's compiled
sig_selective for all 1084 units. The counts are not judged, as the data hold 34 of the 39
sessions.

### fig2-preparatory-more-in-alm

> Preparatory activity (selectivity before the response epoch) is more prevalent in ALM (59% of
> neurons) than in the CN (35%).

Criterion: sample or delay selectivity; denominators (a) all neurons, (b) selective neurons.
The CN neurons are not in either dataset; the criterion says the ALM fraction alone is tested
where the CN neurons are missing (0.59 ± 0.059).

In [ ]:
print(summ[["neurons", "selective", "preparatory", "prep_of_all", "prep_of_selective"]].round(3).to_string())

**Result: partially reproduced.** The ALM fraction is within 0.59 +- 0.059 with all neurons as
denominator (0.563 with correct trials, 0.538 with all trials), not with selective neurons (0.706 to
0.727). The CN fraction, and so the comparison the claim makes, cannot be computed.

### fig2-peri-movement-equal

> Peri-movement activity (selectivity in the response epoch) is equally prevalent in ALM (67%)
> and the CN (66%).

Criterion: response-epoch selectivity; both fractions within tolerance under one reading. Only
the ALM half (0.67 ± 0.067) can be computed.

In [ ]:
print(summ[["neurons", "selective", "peri", "peri_of_all", "peri_of_selective"]].round(3).to_string())

**Result: partially reproduced.** The ALM fraction is 0.642 (correct trials, 1.3 s) and 0.636
(1.5 s), within 0.67 +- 0.067; with all trials 0.562 and 0.571, outside. The CN half, and so the
equality, cannot be tested.

### fig2cd-selectivity-emerges-gradually

> Selectivity for "lick left" and "lick right" trials emerges gradually in both the ALM and the
> CN population averages, starting during the sample epoch and peaking in the response epoch.

Criterion: population selectivity E3 (preferred type from 10 screening trials of each type,
correct control trials, neurons with more than 15 trials of each type, as the Fig. 2 script)
over (a) all selective neurons, (b) neurons with sample or delay selectivity. In a region,
reproduced when the bootstrap 95% interval over neurons excludes zero in at least one window of
the sample epoch, the delay mean exceeds the sample mean, and the maximum lies in the response
epoch. Only ALM can be tested.

In [ ]:
rng = np.random.default_rng(21)
A = sel[(sel.trials == "correct") & (sel.response_s == 1.3)].set_index(["session", "unit"])
curves, prep_flag = [], []
for s in sessions:
    if s.group == "inhibition":
        continue
    t = s.trials
    for u in s.pyramidal():
        row = A.loc[(s.key, u["key"])]
        if not row.selective or row.nR <= 15 or row.nL <= 15:
            continue
        m = s.base_mask() & (t.ptype == 0).to_numpy() & s.stable(u, len(t)) & t.correct.to_numpy()
        r = G.pref_selectivity(s, u, m, {}, 10, rng)
        if r is None:
            continue
        curves.append(r[0]); prep_flag.append(bool(row.preparatory))
curves, prep_flag = np.array(curves), np.array(prep_flag)
g = G.GRID
samp, dl, resp = (g > -2.6) & (g < -1.3), (g > -1.3) & (g < 0), (g > 0) & (g < 1.3)
for name, X in [("(a) all selective", curves), ("(b) sample or delay selective", curves[prep_flag])]:
    m_, lo, hi = G.boot_mean_ci(X, rng, 10000)
    print(f"{name}: n={len(X)}; sample windows with interval above 0: {int((lo[samp] > 0).sum())} of {int(samp.sum())}, "
          f"first at {g[samp][lo[samp] > 0][0] if (lo[samp] > 0).any() else np.nan:.2f} s; "
          f"mean sample {m_[samp].mean():.2f}, delay {m_[dl].mean():.2f}, response {m_[resp].mean():.2f} spikes/s; "
          f"maximum {m_[(g > -3) & (g < 1.8)].max():.2f} at {g[(g > -3) & (g < 1.8)][np.argmax(m_[(g > -3) & (g < 1.8)])]:.2f} s from the go cue")
    print(f"   mean before sample onset (-3.0 to -2.6 s): {m_[(g > -3) & (g < -2.6)].mean():.2f}")
fig, ax = plt.subplots(figsize=(6, 3))
m_, lo, hi = G.boot_mean_ci(curves, rng, 2000)
ax.fill_between(g, lo, hi, color="0.8"); ax.plot(g, m_, "k")
for x in (-2.6, -1.3, 0):
    ax.axvline(x, ls=":", c="0.5")
ax.set_xlim(-3, 1.8); ax.set_xlabel("time from go cue (s)"); ax.set_ylabel("selectivity (spikes/s)")
print(f"plotted range: {np.nanmin(lo[(g > -3) & (g < 1.8)]):.2f} to {np.nanmax(hi[(g > -3) & (g < 1.8)]):.2f}")
plt.show()

**Result: partially reproduced.** For ALM under reading (a), 827 neurons, all three conditions hold:
the interval excludes zero in 117 of 129 sample-epoch points, the delay mean (1.80 spikes/s) exceeds
the sample mean (0.65), and the maximum (2.37) is 0.23 s after the go cue. Under reading (b) the
maximum is 0.06 s before the go cue. The CN half cannot be tested.

## ALM responses to CN photostimulation (Figs. 3 and 4)

Modulation E4: two-tailed t-test of spike counts, control against photostimulation trials, both
trial types pooled, p < 0.01; neurons with at least 8 control trials of each type and more than 3
photostimulation trials of each type (deposited FR script). Windows: (a) the photostimulation
period (0.455 s for photo-activation, 1.3 s for code 6), (b) the whole delay epoch. Trials:
all (correct, error and no-lick, as the FR script) and correct only. Latency E5: 10 ms bins in
1 ms steps from delay onset, sign-flipped for suppressed neurons, first bin with p < 0.01,
10,000 bootstrap resamples of neurons; the bin can be labelled by its start, centre or end
(not stated in the paper), and all three are shown.

In [ ]:
t0 = time.time()
mod = {}
fine = {}
for grp, code_, dur in [("fastigial", 2, 0.455), ("dentate", 4, 0.455), ("inhibition", 6, 1.3)]:
    for which in ("all", "correct"):
        rows, F = [], []
        for s in sessions:
            if s.group != grp:
                continue
            for u in s.pyramidal():
                rd = G.modulation(s, u, code_, 0, "cue", which)
                if rd is None:
                    continue
                rp = G.modulation(s, u, code_, 0, dur, which)
                st_, fc = G.fine_change(s, u, code_, which)
                rows.append(dict(session=s.key, mouse=s.mouse, unit=u["key"], p_delay=rd["p"], change_delay=rd["change"],
                                 rate_ctrl=rd["rate_ctrl"], rate_stim=rd["rate_stim"], p_stim=rp["p"], change_stim=rp["change"]))
                F.append(fc)
        mod[(grp, which)] = pd.DataFrame(rows)
        fine[(grp, which)] = (st_, np.array(F))
n_pyr = {g_: sum(len(s.pyramidal()) for s in sessions if s.group == g_) for g_ in ("fastigial", "dentate", "inhibition")}
print("pyramidal neurons per group:", n_pyr)
tabm = []
for (grp, which), d in mod.items():
    for w in ("stim", "delay"):
        sig = d[f"p_{w}"] < 0.01
        tabm.append(dict(group=grp, trials=which, window=w, tested=len(d), modulated=int(sig.sum()),
                         frac_of_tested=sig.mean(), frac_of_pyramidal=sig.sum() / n_pyr[grp],
                         increase=int((sig & (d[f"change_{w}"] > 0)).sum()), decrease=int((sig & (d[f"change_{w}"] < 0)).sum())))
tabm = pd.DataFrame(tabm)
print(tabm.round(3).to_string(index=False))
print(f"({time.time() - t0:.0f} s)")

In [ ]:
t0 = time.time()
lat_rows = []
for (grp, which), d in mod.items():
    starts, F = fine[(grp, which)]
    for w in ("stim", "delay"):
        sig = (d[f"p_{w}"] < 0.01).to_numpy()
        signs = np.sign(d[f"change_{w}"].to_numpy())[sig]
        L = G.latency(F[sig], signs, np.random.default_rng(31), n_boot=10000, starts=starts)
        for lab, sh in (("start", 0.0), ("centre", 5.0), ("end", 10.0)):
            lat_rows.append(dict(group=grp, trials=which, window=w, label=lab, n=L["n"],
                                 onset_all_ms=L["onset_all"] + sh, boot_mean_ms=L["boot_mean"] + sh,
                                 boot_sd_ms=L["boot_sd"], no_onset=L["no_onset"]))
latd = pd.DataFrame(lat_rows)
print(latd.round(2).to_string(index=False))
print(f"({time.time() - t0:.0f} s)")

### fig4-fastigial-activation-alm-latency

> Fastigial photo-activation changes ALM activity rapidly (latency 4.5 plus or minus 2.6 ms),
> consistent with a disynaptic pathway through the thalamus.

Criterion: latency within 4.5 ± 2.6 ms under one reading, and the count of modulated neurons
within 227 ± 22.7, as the dataset holds all 6 mice.

In [ ]:
print(tabm[tabm.group == "fastigial"].round(3).to_string(index=False))
print(latd[latd.group == "fastigial"].round(2).to_string(index=False))
d = mod[("fastigial", "all")]
print("\nModulated (stimulation window, all trials) by mouse:")
print(d.assign(sig=d.p_stim < 0.01).groupby("mouse").sig.agg(["size", "sum"]).to_string())
# variant: no minimum trial count beyond two trials of each condition
cnt = 0; tested = 0
for s in fa:
    for u in s.pyramidal():
        ctrl, stim, _ = G.stim_trials(s, u, 2, "all")
        if ctrl.sum() < 2 or stim.sum() < 2:
            continue
        tested += 1
        po = s.trials.pole_out.to_numpy()
        c = s.counts(u, po, po + 0.455)
        if np.var(c[ctrl]) + np.var(c[stim]) > 0 and stats.ttest_ind(c[ctrl], c[stim]).pvalue < 0.01:
            cnt += 1
print(f"\nVariant without the script's minimum trial counts (stimulation window, all trials): {cnt} of {tested}")
keep = ~d.session.isin(MISMATCH).to_numpy()
starts, F = fine[("fastigial", "all")]
sig = (d.p_stim < 0.01).to_numpy() & keep
L = G.latency(F[sig], np.sign(d.change_stim.to_numpy())[sig], np.random.default_rng(32), n_boot=10000, starts=starts)
print(f"Without the sessions in MISMATCH (stimulation window, all trials): tested {int(keep.sum())}, modulated {int(sig.sum())}; "
      f"latency by bin start {L['boot_mean']:.2f} +- {L['boot_sd']:.2f} ms, by centre {L['boot_mean'] + 5:.2f}")

**Result: partially reproduced.** The latency is 3.64 +- 2.91 ms with the stimulation window, all
trials and bins labelled by their start, within 4.5 +- 2.6; labelled by centre it is 8.64 ms,
outside. The start label gives the earliest time an effect could begin in its bin. The count, 177
of 445 tested neurons (187 with the whole delay, 67 to 68 with correct trials, 190 without the
minimum trial counts), is outside 227 +- 22.7 although all 6 mice are present. Without the sessions
in MISMATCH: 166 modulated, 4.22 ms by bin start.

### fig4g-dentate-effect-recovers

> Dentate photo-activation also changes ALM responses (latency 8.3 plus or minus 5.0 ms), but ALM
> activity recovers quickly after it.

Criterion: latency within 8.3 ± 5.0 ms and count within 163 ± 16.3 under one reading, and the
fraction of significant neurons in bins starting 1 s or more after photostimulus offset not
above the comparison of fig4f (p >= 0.05) and lower than the fastigial fraction (one-sided test
of two proportions, p < 0.05). The persistence numbers are printed in the fig4f section.

In [ ]:
print(tabm[tabm.group == "dentate"].round(3).to_string(index=False))
print(latd[latd.group == "dentate"].round(2).to_string(index=False))
d = mod[("dentate", "all")]
keep = ~d.session.isin(MISMATCH).to_numpy()
starts, F = fine[("dentate", "all")]
sig = (d.p_stim < 0.01).to_numpy() & keep
L = G.latency(F[sig], np.sign(d.change_stim.to_numpy())[sig], np.random.default_rng(33), n_boot=10000, starts=starts)
print(f"\nWithout the sessions in MISMATCH (stimulation window, all trials): tested {int(keep.sum())}, modulated {int(sig.sum())}; "
      f"latency by bin start {L['boot_mean']:.2f} +- {L['boot_sd']:.2f} ms, by centre {L['boot_mean'] + 5:.2f}")
print("Modulated (stimulation window, all trials) by mouse:")
print(d.assign(sig=d.p_stim < 0.01).groupby("mouse").sig.agg(["size", "sum"]).to_string())
r_ = latd[(latd.group == "dentate") & (latd.trials == "all") & (latd.window == "stim") & (latd.label == "start")].iloc[0]
print(f"\nStimulation window, all trials, bins labelled by start: latency {r_.boot_mean_ms:.4f} +- {r_.boot_sd_ms:.4f} ms; "
      f"within 8.3 +- 5.0 (3.3 to 13.3): {bool(3.3 <= r_.boot_mean_ms <= 13.3)}")

### fig4f-fastigial-effect-persists

> Transient fastigial photo-activation alters ALM preparatory activity long (more than 1 s)
> after the photostimulus ends.

Criterion: E4 in 400 ms bins in 50 ms steps (Methods), fraction of neurons with p < 0.01 in
bins starting 1 s or more after photostimulus offset (offset at 0.845 s before the go cue, so
these bins lie in the response epoch); compared with (a) 1% by a one-sided binomial test, (b)
the fraction in bins ending before photostimulus onset by a one-sided test of two proportions.
The mean fraction over those bins is tested, with the number of neurons as n. Neurons and trials
as the FR script (all trials).

In [ ]:
t0 = time.time()
tb = np.round(np.arange(-3, 2.0001, 0.05), 3)      # bin centres relative to the go cue (FR script)
pre = (tb + 0.2 <= -1.3) & (tb - 0.2 >= -3.0)
post = (tb - 0.2 >= -1.3 + 0.455 + 1.0) & (tb - 0.2 < 1.3)


def tp_(a, b):
    return stats.ttest_ind(a, b).pvalue if np.var(a) + np.var(b) > 0 else 1.0


def persistence_test(P):
    """Mean fraction significant in the post bins, against 1% and against the pre-onset bins."""
    f = (P < 0.01).mean(0)
    n = len(P)
    fpost, fpre = f[post].mean(), f[pre].mean()
    k, kp = int(round(fpost * n)), int(round(fpre * n))
    pa = stats.binomtest(k, n, 0.01, alternative="greater").pvalue
    z = sm.stats.proportions_ztest([k, kp], [n, n], alternative="larger")
    return f, n, fpost, fpre, k, pa, z


persist, persist_keys, persist_dir = {}, {}, {}
for grp, code_, ss in [("fastigial", 2, fa), ("dentate", 4, de)]:
    P, K, PD = [], [], {"R": [], "L": []}
    for s in ss:
        cue = s.trials.cue.to_numpy()
        lk = s.trials.licked.to_numpy()
        for u in s.pyramidal():
            ctrl, stim, en = G.stim_trials(s, u, code_, "all")
            if not en:
                continue
            C = np.array([s.counts(u, cue + b_ - 0.2, cue + b_ + 0.2) for b_ in tb])
            P.append([tp_(c[ctrl], c[stim]) for c in C]); K.append((s.key, u["key"]))
            for dd in "RL":      # within the licked direction (variant)
                c_, s_ = ctrl & (lk == dd), stim & (lk == dd)
                if c_.sum() >= 8 and s_.sum() > 3:
                    PD[dd].append([tp_(c[c_], c[s_]) for c in C])
    persist[grp], persist_keys[grp] = np.array(P), K
    persist_dir[grp] = {dd: np.array(v) for dd, v in PD.items()}
res_p = {}
for grp, P in persist.items():
    f, n, fpost, fpre, k, pa, z = persistence_test(P)
    res_p[grp] = (fpost, n, k)
    print(f"{grp}: neurons {n}; fraction significant: before onset {fpre:.4f}, during light max {f[(tb > -1.3) & (tb < -0.845)].max():.3f}, "
          f">=1 s after offset {fpost:.4f} ({post.sum()} bins); (a) vs 1%: binomial p = {pa:.2g}; (b) vs before onset: z = {z[0]:.2f}, p = {z[1]:.2g}")
    print("   fraction per bin (every 4th bin, centre in s from go cue):",
          " ".join(f"{b_:.2f}:{x:.3f}" for b_, x in zip(tb[::4], f[::4])))
(ff, nf, kf), (fd, nd, kd) = res_p["fastigial"], res_p["dentate"]
z = sm.stats.proportions_ztest([kd, kf], [nd, nf], alternative="smaller")
print(f"dentate lower than fastigial after offset: {fd:.4f} vs {ff:.4f}, z = {z[0]:.2f}, one-sided p = {z[1]:.2g}")

# the same p-values in the lab's FR compile (p_stim2_FN, p_stim2_DN), neuron by neuron
for grp, lab_key in [("fastigial", "p_stim2_FN"), ("dentate", "p_stim2_DN")]:
    lab_p = np.asarray(lab_fr[lab_key])[[pa_row[k_] for k_ in persist_keys[grp]]]
    print(f"{grp}: largest difference from the lab's compiled p-values over {len(lab_p)} neurons x {len(tb)} bins: "
          f"{np.nanmax(np.abs(lab_p - persist[grp])):.1e}; NaN in the lab's rows: {int(np.isnan(lab_p).sum())} "
          f"(not compared; the values here at those entries: {pd.Series(persist[grp][np.isnan(lab_p)]).round(6).value_counts().to_dict()})")

print("\nVariant without the sessions in MISMATCH:")
res_m = {}
for grp in persist:
    keep = np.array([k_[0] not in MISMATCH for k_ in persist_keys[grp]])
    f, n, fpost, fpre, k, pa, z = persistence_test(persist[grp][keep])
    res_m[grp] = (fpost, n, k)
    print(f"  {grp}: neurons {n}; before onset {fpre:.4f}, >=1 s after offset {fpost:.4f}; vs 1% p = {pa:.2g}; vs before z = {z[0]:.2f}, p = {z[1]:.2g}")
(ff, nf, kf), (fd, nd, kd) = res_m["fastigial"], res_m["dentate"]
z = sm.stats.proportions_ztest([kd, kf], [nd, nf], alternative="smaller")
print(f"  dentate lower than fastigial: one-sided p = {z[1]:.2g}")

print("\nVariant within the licked direction (control against photo-activation trials with the same lick; "
      "neurons with >= 8 control and > 3 photo-activation trials of that direction):")
res_d = {}
for grp in persist_dir:
    for dd in "RL":
        f, n, fpost, fpre, k, pa, z = persistence_test(persist_dir[grp][dd])
        res_d[(grp, dd)] = (fpost, n, k)
        print(f"  {grp}, licked {dd}: neurons {n}; before onset {fpre:.4f}, >=1 s after offset {fpost:.4f}; vs 1% p = {pa:.2g}; "
              f"vs before z = {z[0]:.2f}, p = {z[1]:.2g}")
for dd in "RL":
    (ff, nf, kf), (fd, nd, kd) = res_d[("fastigial", dd)], res_d[("dentate", dd)]
    z = sm.stats.proportions_ztest([kd, kf], [nd, nf], alternative="smaller")
    print(f"  licked {dd}: dentate lower than fastigial, one-sided p = {z[1]:.2g}")
print(f"({time.time() - t0:.0f} s)")

**Result: reproduced.** In bins starting 1 s or more after light offset a mean 0.1345 of 445
neurons change their rate, above 1% (p = 3.3e-47) and above the 0.0092 before light onset
(p = 1.8e-13). It holds without the sessions in MISMATCH (0.1457) and within each licked direction
(0.127 for "lick right", 0.072 for "lick left"; against before onset p = 3.1e-10 and 0.0016), so the
different mix of licks after photo-activation does not explain it. These 400 ms-bin p-values equal
the lab's compiled ones (p_stim2_FN, p_stim2_DN) to within 3e-13 wherever the lab's file has a value.
Its 381 and 704 NaN entries are 1.0 here, so neither counts them significant; this notebook scores
1.0 where neither set of trials varies, where MATLAB's t-test gives NaN.

**Result: partially reproduced.** With the stimulation window and all trials, 172 modulated neurons
and the latency are within tolerance: 3.3022 +- 3.07 ms with bins labelled by their start, the label
used for fig4-fastigial-activation-alm-latency, just inside 3.3 to 13.3 (printed), and 8.30 ms by
centre. The start label thus matches both the fastigial and the dentate latency, the dentate one at
the lower bound. The recovery part
fails: 0.0476 of neurons are significant 1 s or more after offset, above 1% (p = 2.2e-10) and above
the 0.0147 before onset (p = 0.0013), though below fastigial (0.1345, p = 1.2e-6). Without the
sessions in MISMATCH it stays above both (0.0475; p = 6.1e-9 and 0.0007). Within the licked
direction it is above 1% for both directions (0.036 and 0.027) and above before onset only for
"lick right" (p = 0.024; "lick left" p = 0.098).

### fig3d-alm-altered-by-cn-inhibition

> CN photo-inhibition during the delay rapidly alters the activity of many ALM neurons (197 of
> 454), with a latency of 7.1 plus or minus 3.8 ms.

Criterion: 1.3 s delay photo-inhibition (code 6), E4 on the whole delay epoch, latency E5 (both
window readings); reproduced when the count (or the fraction 0.43 where the dataset holds fewer
neurons) and the latency are within tolerance under one reading.

In [ ]:
print(tabm[tabm.group == "inhibition"].round(3).to_string(index=False))
print(latd[latd.group == "inhibition"].round(2).to_string(index=False))
d = mod[("inhibition", "all")]
print("\nBy mouse (delay window, all trials):")
print(d.assign(sig=d.p_delay < 0.01).groupby("mouse").sig.agg(["size", "sum", "mean"]).round(3).to_string())

**Result: not reproduced.** 228 of 486 tested pyramidal neurons change their delay rate (106 of 263
with correct trials). The count is outside 197 +- 19.7, and the latency, 15.3 +- 12.9 ms (bin
start, all trials) to 41.7 ms, is never within 7.1 +- 3.8. The fraction, 0.469, is within 10% of
0.43, but the criterion uses it only where the dataset holds fewer neurons, and here it holds more
(682 pyramidal neurons against 454).

### fig3e-alm-mean-rate-unchanged

> Despite the changes in individual neurons, the average ALM spike rate is nearly unchanged by
> CN photo-inhibition (4.15 against 4.39 spikes/s).

Criterion: a claim of absence on the paper's two-tailed t-test; mean delay rate per neuron in
control and photo-inhibition trials (E4 trials), neurons (a) all tested pyramidal, (b)
modulated; t-test paired and unpaired. Reproduced when the rates are within tolerance under one
reading of the neurons and the t-test finds no difference.

In [ ]:
for which in ("all", "correct"):
    d = mod[("inhibition", which)]
    for nm, dd in [("(a) all tested", d), ("(b) modulated", d[d.p_delay < 0.01])]:
        print(f"trials {which}, {nm}: n={len(dd)}, control {dd.rate_ctrl.mean():.2f}, inhibition {dd.rate_stim.mean():.2f} spikes/s, "
              f"change {dd.rate_stim.mean() - dd.rate_ctrl.mean():+.2f}; paired p = {stats.ttest_rel(dd.rate_ctrl, dd.rate_stim).pvalue:.3f}, "
              f"unpaired p = {stats.ttest_ind(dd.rate_ctrl, dd.rate_stim).pvalue:.3f}")
d = mod[("inhibition", "all")]
print("\nBy mouse, all tested neurons, all trials (control, inhibition, spikes/s):")
print(d.groupby("mouse")[["rate_ctrl", "rate_stim"]].mean().round(2).to_string())

**Result: reproduced.** Over all 486 tested neurons and all trials the delay rate is 4.03 spikes/s
in control and 4.21 with CN photo-inhibition, within 4.15 +- 0.415 and 4.39 +- 0.439, with no
difference (paired p = 0.51, unpaired p = 0.66).

### fig3e-alm-increase-decrease

> Among ALM neurons modulated by CN photo-inhibition, 108 increase and 89 decrease their
> responses.

Criterion: sign of the delay rate change among the neurons of fig3d; both counts within
tolerance, or, where the dataset holds fewer neurons, the increase fraction 0.55 within 10%.

In [ ]:
x = tabm[(tabm.group == "inhibition")]
x = x.assign(increase_fraction=x.increase / x.modulated)
print(x[["trials", "window", "tested", "modulated", "increase", "decrease", "increase_fraction"]].round(3).to_string(index=False))

**Result: not reproduced.** Of the modulated neurons 122 increase and 106 decrease (all trials), 65
and 41 (correct trials), outside 108 +- 10.8 and 89 +- 8.9. The increase fraction with all trials,
0.535, is within 10% of 0.55, but the criterion uses it only where the dataset holds fewer neurons.

## ALM selectivity under CN photostimulation

Population selectivity E3 over selective neurons (E2, correct trials, 1.3 s response window)
tested on more than 3 trials in every condition (control trials beyond the screening trials,
and photostimulation trials, of each type), all trials grouped by instructed type (Extended
Data Figs. 5b and 9c legends). Preferred type from 10 screening control trials of each type
(Methods); the deposited photostimulation scripts use 5, shown as a variant.

In [ ]:
t0 = time.time()
selstim = {}
for grp, code_ in [("inhibition", 6), ("fastigial", 2), ("dentate", 4)]:
    for nscr in (10, 5):
        rng = np.random.default_rng(41 + nscr)
        C, Z, keys = [], [], []
        for s in sessions:
            if s.group != grp:
                continue
            t = s.trials
            n = len(t)
            R = (t.instructed == "R").to_numpy(); L = (t.instructed == "L").to_numpy()
            for u in s.pyramidal():
                e2 = G.selectivity_tests(s, u, "correct")
                if not any(e2[e] < 0.05 for e in ("sample", "delay", "response") if not np.isnan(e2[e])):
                    continue
                base = s.base_mask() & s.stable(u, n) & (t.correct | t.error).to_numpy()
                ctrl = base & (t.ptype == 0).to_numpy(); stim = base & (t.ptype == code_).to_numpy()
                if min((ctrl & R).sum(), (ctrl & L).sum()) - nscr <= 3 or min((stim & R).sum(), (stim & L).sum()) <= 3:
                    continue
                r = G.pref_selectivity(s, u, ctrl, {"stim": stim}, nscr, rng)
                C.append(r[0]); Z.append(r[1]["stim"]); keys.append((s.key, u["key"]))
        selstim[(grp, nscr)] = (np.array(C), np.array(Z), keys)
g = G.GRID
W = {"whole delay": (g > -1.3) & (g < 0), "last 500 ms of delay": (g > -0.5) & (g < 0),
     "light offset to go cue": (g > -1.3 + 0.455) & (g < 0)}
rows = []
for (grp, nscr), (C, Z, keys) in selstim.items():
    rng = np.random.default_rng(50)
    for wn, w in W.items():
        if grp == "inhibition" and wn == "light offset to go cue":
            continue
        c, z = np.nanmean(C[:, w], 1), np.nanmean(Z[:, w], 1)
        mc, lc, hc = G.boot_mean_ci(c, rng); mz, lz, hz = G.boot_mean_ci(z, rng); md_, ld, hd = G.boot_mean_ci(c - z, rng)
        rows.append(dict(group=grp, screen=nscr, window=wn, neurons=len(C), control=mc, control_lo=lc, control_hi=hc,
                         stim=mz, stim_lo=lz, stim_hi=hz, diff=md_, diff_lo=ld, diff_hi=hd))
selres = pd.DataFrame(rows)
print(selres.round(2).to_string(index=False))
print(f"({time.time() - t0:.0f} s)")

### fig3f-alm-selectivity-abolished-by-cn-inhibition

> CN photo-inhibition abolishes ALM trial-type selectivity.

Criterion: control selectivity above zero and photo-inhibition selectivity not distinguishable
from zero (its 95% interval includes zero), windows (a) whole delay, (b) last 500 ms.

In [ ]:
print(selres[selres.group == "inhibition"].round(2).to_string(index=False))
C, Z, keys = selstim[("inhibition", 10)]
print("neurons by mouse:", pd.Series([k[0].split("_")[0] for k in keys]).value_counts().to_dict())
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(G.GRID, np.nanmean(C, 0), "k", label="control"); ax.plot(G.GRID, np.nanmean(Z, 0), "c", label="CN photo-inhibition")
for x_ in (-2.6, -1.3, 0):
    ax.axvline(x_, ls=":", c="0.5")
ax.set_xlim(-3, 1.6); ax.legend(); ax.set_xlabel("time from go cue (s)"); ax.set_ylabel("selectivity (spikes/s)")
sl = (G.GRID > -3) & (G.GRID < 1.6)
print(f"plotted range: {min(np.nanmean(C, 0)[sl].min(), np.nanmean(Z, 0)[sl].min()):.2f} to {max(np.nanmean(C, 0)[sl].max(), np.nanmean(Z, 0)[sl].max()):.2f}")
plt.show()

**Result: partially reproduced.** Selectivity falls from 1.46 to 0.39 spikes/s over the whole delay
and from 1.52 to 0.27 in its last 500 ms, but the photo-inhibition intervals ([0.19, 0.60] and [0.02,
0.53]) exclude zero, also with 5 screening trials ([0.17, 0.57] and [0.06, 0.56]). It is reduced by
73% to 82%, not abolished. The verified outcome is kept; a change to not reproduced is proposed in
the attempt record, as the criterion's only test, that photo-inhibition selectivity cannot be told
from zero, fails in both windows and with both screening counts (printed above).

### fig4e-fastigial-activation-abolishes-alm-selectivity

> Fastigial photo-activation abolishes selective persistent and ramping activity in most ALM
> neurons, while in a minority it creates new preparatory activity.

Criterion: n = 328 neurons expected; population selectivity lower in photo-activation trials
(interval of the difference excludes zero) in the delay after the photostimulus ((a) light
offset to go cue, (b) last 500 ms); per neuron, the t-test of E2 on that window (all trials by
instructed type), control against photo-activation trials: more than half of the
control-selective neurons lose significance, and fewer gain it than lose it.

In [ ]:
print(selres[selres.group == "fastigial"].round(2).to_string(index=False))


def lose_gain(grp, code_, keys, rng, n_sub=100):
    """Per neuron, E2 t-test on the window in control and photo-activation trials (instructed type),
    with all control trials and with control trials subsampled to the photo-activation counts of
    each type (mean over n_sub subsamples), which removes the difference in power."""
    kset = set(keys)
    out = []
    for wn, (a_rel, b_rel) in {"(a) light offset to go cue": (0.455, None), "(b) last 500 ms": (None, 0.5)}.items():
        for s in sessions:
            if s.group != grp:
                continue
            t = s.trials
            n = len(t)
            R = (t.instructed == "R").to_numpy(); L = (t.instructed == "L").to_numpy()
            po, cue = t.pole_out.to_numpy(), t.cue.to_numpy()
            a = po + a_rel if a_rel is not None else cue - b_rel
            for u in s.pyramidal():
                if (s.key, u["key"]) not in kset:
                    continue
                base = s.base_mask() & s.stable(u, n) & (t.correct | t.error).to_numpy()
                ctrl = base & (t.ptype == 0).to_numpy(); stim = base & (t.ptype == code_).to_numpy()
                c = s.counts(u, a, cue)
                cR, cL, sR, sL = c[ctrl & R], c[ctrl & L], c[stim & R], c[stim & L]
                pc, ps = tp_(cR, cL) < 0.05, tp_(sR, sL) < 0.05
                sub = np.array([tp_(rng.choice(cR, len(sR), replace=False), rng.choice(cL, len(sL), replace=False)) < 0.05
                                for _ in range(n_sub)])
                out.append(dict(window=wn, session=s.key, ctrl_sel=pc, lose=pc and not ps, gain=ps and not pc,
                                sub_sel=sub.mean(), sub_lose=(sub & (not ps)).mean(), sub_gain=(~sub & ps).mean()))
    return pd.DataFrame(out)


rng = np.random.default_rng(56)
lg = {grp: lose_gain(grp, code_, selstim[(grp, 10)][2], rng) for grp, code_ in [("fastigial", 2), ("dentate", 4)]}
for grp, x in lg.items():
    for lab_, xx in [("all sessions", x), ("without MISMATCH", x[~x.session.isin(MISMATCH)])]:
        for wn, g_ in xx.groupby("window"):
            print(f"{grp}, {lab_}, {wn}: neurons {len(g_)}; all control trials: selective in control {int(g_.ctrl_sel.sum())}, "
                  f"lose significance {int(g_.lose.sum())} ({g_.lose.sum() / max(g_.ctrl_sel.sum(), 1):.2f}), gain {int(g_.gain.sum())}; "
                  f"control subsampled to the photo-activation counts: selective {g_.sub_sel.sum():.1f}, lose {g_.sub_lose.sum():.1f} "
                  f"({g_.sub_lose.sum() / g_.sub_sel.sum():.2f}), gain {g_.sub_gain.sum():.1f}")
C, Z, keys = selstim[("fastigial", 10)]
keep = np.array([k_[0] not in MISMATCH for k_ in keys])
rng = np.random.default_rng(57)
for wn, w in [("light offset to go cue", W["light offset to go cue"]), ("last 500 ms of delay", W["last 500 ms of delay"])]:
    c, z = np.nanmean(C[keep][:, w], 1), np.nanmean(Z[keep][:, w], 1)
    md_, ld, hd = G.boot_mean_ci(c - z, rng)
    print(f"Population selectivity without MISMATCH ({int(keep.sum())} neurons), {wn}: control {c.mean():.2f}, "
          f"photo-activation {z.mean():.2f}, difference {md_:.2f} [{ld:.2f}, {hd:.2f}]")

**Result: reproduced.** 328 neurons enter, as in the paper. From light offset to the go cue
selectivity is 2.20 spikes/s in control and 0.66 with photo-activation (difference 1.55 [1.01, 2.09]);
163 of the 183 control-selective neurons lose significance and 6 gain it (last 500 ms: 157 of 174,
and 9). With control trials subsampled to the photo-activation counts, 0.86 and 0.89 of the
control-selective neurons still lose it and 16.3 and 18.6 gain it, against 0.51 and 0.52 losing and
35.4 and 30.1 gaining with dentate photo-activation, so the loss is not a matter of fewer trials.
Without the sessions in MISMATCH, 151 of 165 lose it and 6 gain it.

## Coding direction (Fig. 4h, 4i; Extended Data Fig. 10)

E6: sessions with more than 5 neurons and more than 5 trials per condition. Neurons are the
pyramidal units stable from the 50th trial to the 50th before the end, and trials those inside
every neuron's stable range, as the deposited CD script; rates in 400 ms windows in 10 ms
steps, labelled by window centre. Control trials are split at random into halves: correct trials
of the first half estimate the cd (mean "lick right" minus mean "lick left" over windows centred
in the delay, unit length) and the session offset; the second half and the photostimulation
trials (correct and error, by instructed type) are projected. Bootstrap over sessions.

In [ ]:
rng = np.random.default_rng(60)
cds = []
skipped = []
for s in sessions:
    if s.group == "inhibition":
        continue
    r = G.cd_session(s, rng, stim_code=2 if s.group == "fastigial" else 4)
    (cds.append(r) if r is not None else skipped.append(s.key))
print(f"sessions used: {len(cds)} of {len(cds) + len(skipped)} "
      f"(fastigial {sum(r['group'] == 'fastigial' for r in cds)}, dentate {sum(r['group'] == 'dentate' for r in cds)}); "
      f"neurons per session {min(r['n_neurons'] for r in cds)} to {max(r['n_neurons'] for r in cds)}")
print("not used:", skipped)
cent = cds[0]["centres"]


def separation(r, a, b):
    return r["proj"][r["sets"][a]].mean(0) - r["proj"][r["sets"][b]].mean(0)
print("in the lab's CD compile but not used here:", sorted(set(lab_cd_sessions) - {r["session"] for r in cds}))
print("used here but not in the lab's CD compile:", sorted({r["session"] for r in cds} - set(lab_cd_sessions)))
# variant: the lab's compile keeps sessions with >= 3 trials in every condition (analysis_compiled_all_ALM_DCNstim_CD.m)
rng_lab = np.random.default_rng(64)
cds_lab = [r for r in (G.cd_session(s, rng_lab, stim_code=2 if s.group == "fastigial" else 4, min_trials=2)
                       for s in sessions if s.group != "inhibition") if r is not None]
print(f"variant with >= 3 trials per condition: {len(cds_lab)} sessions (fastigial {sum(r['group'] == 'fastigial' for r in cds_lab)}, "
      f"dentate {sum(r['group'] == 'dentate' for r in cds_lab)}); differing from the lab's list:",
      sorted({r["session"] for r in cds_lab} ^ set(lab_cd_sessions)))

### fig4i-cd-control-separation

> In control trials, ALM activity trajectories for "lick left" and "lick right" diverge along
> the coding direction during the sample epoch and stay separated through the delay epoch.

Criterion: "lick right" minus "lick left" projection of independent control trials, mean over
sessions with a bootstrap over sessions; reproduced when the 95% interval excludes zero at some
time in the sample epoch and at every window centred within the delay epoch.

In [ ]:
rng = np.random.default_rng(61)
X = np.array([separation(r, "test_R", "test_L") for r in cds])
m_, lo, hi = G.boot_mean_ci(X, rng)
samp = (cent > -2.6) & (cent < -1.3); dl = (cent >= -1.3) & (cent <= 0)
print(f"sessions {len(X)}; sample-epoch windows with interval above 0: {int((lo[samp] > 0).sum())} of {int(samp.sum())} "
      f"(first at {cent[samp][lo[samp] > 0][0]:.2f} s from the go cue)")
print(f"delay windows with interval above 0: {int((lo[dl] > 0).sum())} of {int(dl.sum())}; smallest lower bound {lo[dl].min():.2f}")
print(f"separation before sample onset (centres -3.0 to -2.8 s): {m_[(cent >= -3.0) & (cent <= -2.8)].mean():.2f}")
fig, ax = plt.subplots(figsize=(6, 3))
ax.fill_between(cent, lo, hi, color="0.8"); ax.plot(cent, m_, "k")
for x_ in (-2.6, -1.3, 0):
    ax.axvline(x_, ls=":", c="0.5")
ax.set_xlabel("time from go cue (s)"); ax.set_ylabel("R minus L projection")
print(f"plotted range: {lo.min():.2f} to {hi.max():.2f}")
plt.show()

**Result: reproduced.** Over 20 sessions the interval excludes zero from 2.06 s before the go cue
(75 of 130 sample-epoch windows) and in all 131 windows centred in the delay (smallest lower bound
5.98); before sample onset the separation is -0.13.

### fig4i-fastigial-collapses-cd

> Fastigial photo-activation collapses ALM activity trajectories along the coding direction,
> while dentate photo-activation minimally affects them.

Criterion: separation in control, fastigial and dentate photo-activation trials in the last 400
ms of the delay and averaged from light offset to the go cue; "collapse" read as (a) fastigial
separation not distinguishable from zero, (b) lower than control (interval of the difference
excludes zero); reproduced when, in the same window, the fastigial separation collapses and the
dentate separation does not differ from its control (interval of the difference includes zero).

In [ ]:
rng = np.random.default_rng(62)
for (grp, excl) in [("fastigial", False), ("dentate", False), ("fastigial", True), ("dentate", True)]:
    R_ = [r for r in cds if r["group"] == grp and not (excl and r["session"] in MISMATCH)]
    if excl and grp == "fastigial":
        print("\nWithout the sessions in MISMATCH:")
    cs = np.array([separation(r, "test_R", "test_L") for r in R_]); st_ = np.array([separation(r, "stim_R", "stim_L") for r in R_])
    for wn, w in [("last 400 ms", np.isclose(cent, -0.2)), ("light offset to go cue", (cent >= -1.3 + 0.455) & (cent <= 0))]:
        a, b = cs[:, w].mean(1), st_[:, w].mean(1)
        ma, la, ha = G.boot_mean_ci(a, rng); mb, lb, hb = G.boot_mean_ci(b, rng); md_, ld, hd = G.boot_mean_ci(a - b, rng)
        print(f"{grp} ({len(R_)} sessions), {wn}: control {ma:.2f} [{la:.2f}, {ha:.2f}], photo-activation {mb:.2f} [{lb:.2f}, {hb:.2f}], "
              f"control minus stim {md_:.2f} [{ld:.2f}, {hd:.2f}] ({md_ / ma:.0%} of control)")
        if wn == "last 400 ms" and not excl:
            print("   per session (control, photo-activation):",
                  "; ".join(f"{r['session']} {x:.1f}, {y:.1f}" for r, x, y in zip(R_, a, b)))
print("\nVariant with >= 3 trials per condition (cds_lab), last 400 ms:")
for grp in ("fastigial", "dentate"):
    R_ = [r for r in cds_lab if r["group"] == grp]
    a = np.array([separation(r, "test_R", "test_L") for r in R_])[:, np.isclose(cent, -0.2)].mean(1)
    b = np.array([separation(r, "stim_R", "stim_L") for r in R_])[:, np.isclose(cent, -0.2)].mean(1)
    ma, la, ha = G.boot_mean_ci(a, rng); mb, lb, hb = G.boot_mean_ci(b, rng); md_, ld, hd = G.boot_mean_ci(a - b, rng)
    print(f"{grp} ({len(R_)} sessions): control {ma:.2f} [{la:.2f}, {ha:.2f}], photo-activation {mb:.2f} [{lb:.2f}, {hb:.2f}], "
          f"control minus stim {md_:.2f} [{ld:.2f}, {hd:.2f}]")

**Result: partially reproduced.** In the last 400 ms the fastigial separation falls from 18.39 to
6.09 (difference [5.84, 20.91]), a collapse under reading (b) but not (a) (interval [0.26, 14.46]).
The dentate difference, 2.07 [0.15, 4.18], excludes zero, so the dentate condition fails; also from
light offset ([0.75, 4.65]) and with the lab's session rule, 11 fastigial and 12 dentate sessions
(fastigial photo-activation [2.31, 11.88], dentate difference [0.26, 4.24]). Without the sessions in
MISMATCH both conditions hold in the last 400 ms (fastigial [-0.67, 5.72], dentate difference [-0.41,
4.25]), but not from light offset (dentate [0.34, 4.97]).

### cd-captures-selectivity

> The projection along the coding direction captures 64.1% of ALM population selectivity over
> the sample and delay epochs and 27.0% of the total variance in task-related activity.

Criterion: per session, control trials, sample and delay epochs; selectivity is the RMS of the
"lick right" minus "lick left" rate difference, variance the RMS of the baseline-subtracted
activity (baseline: the 500 ms before sample onset); captured fraction (a) ratio of RMS along
the unit cd to RMS over all neurons, (b) its square. Reproduced when under one reading both
means are within tolerance. Variants: measures on the cd-estimating trials (in sample) or on the
independent correct control trials; activity variance from condition means or from single
trials.

In [ ]:
rows = []
for r in cds:
    X_, sets, cd = r["X"], r["sets"], r["cd"]
    w = (cent > r["pole_in"]) & (cent <= 0)
    base = (cent < r["pole_in"]) & (cent >= r["pole_in"] - 0.5)
    tc = sessions[[s.key for s in sessions].index(r["session"])].trials.correct.to_numpy()
    for nm, Rr, Ll in [("in sample", sets["train_R"], sets["train_L"]),
                       ("held out", sets["test_R"][tc[sets["test_R"]]], sets["test_L"][tc[sets["test_L"]]])]:
        xr, xl = X_[:, Rr].mean(1), X_[:, Ll].mean(1)
        D = (xr - xl)[:, w]
        sel_ratio = np.sqrt(((cd @ D) ** 2).mean()) / np.sqrt((D ** 2).sum(0).mean())
        b0 = (xr[:, base].mean(1) + xl[:, base].mean(1)) / 2
        A = np.concatenate([xr[:, w], xl[:, w]], 1) - b0[:, None]
        var_ratio = np.sqrt(((cd @ A) ** 2).mean()) / np.sqrt((A ** 2).sum(0).mean())
        T = np.concatenate([X_[:, Rr][:, :, w], X_[:, Ll][:, :, w]], 1) - b0[:, None, None]
        var_trial = np.sqrt((np.einsum("j,jit->it", cd, T) ** 2).mean()) / np.sqrt((T ** 2).sum(0).mean())
        rows.append(dict(session=r["session"], trials=nm, sel_a=sel_ratio * 100, sel_b=sel_ratio ** 2 * 100,
                         var_a=var_ratio * 100, var_b=var_ratio ** 2 * 100, var_trial_a=var_trial * 100, var_trial_b=var_trial ** 2 * 100))
cap = pd.DataFrame(rows)
agg = cap.drop(columns="session").groupby("trials").agg(["mean", "sem", "std"])
print(agg.round(1).T.to_string())

**Result: partially reproduced.** Under reading (b) the cd captures 66.3% (s.e.m. 5.0) of the
selectivity measured on held-out correct trials, within 64.1 +- 3.4 (73.4% on the cd-estimating
trials, outside). The variance of task-related activity, taken as condition-mean activity, gives
35.9% and 38.7% under reading (b) and 56.0% and 58.9% under (a), all outside 27.0 +- 4.2. Only the
variance over single trials, 28.4%, falls within it, and that variant is not a reading of the claims
file.

### ed10d-fastigial-endpoints-no-prediction

> After fastigial photo-activation, the distance of ALM trajectories to the endpoints along the
> coding direction no longer predicts future movements.

Criterion: projection in the last 400 ms of the delay (window centred 0.2 s before the go cue)
of fastigial photo-activation trials, pooled over sessions; logistic regression of lick direction
on the projection; a claim of absence, reproduced when the slope is not significant (p < 0.05).
Control trials (independent half) for comparison. The projection is offset per session (E6);
it is scaled by its standard deviation over the pooled trials for the fit, which does not change
the p-value.

In [ ]:
k = np.isclose(cent, -0.2)
for nm, a, b, excl in [("fastigial photo-activation", "stim_R", "stim_L", False), ("control (independent half)", "test_R", "test_L", False),
                       ("fastigial photo-activation, without MISMATCH", "stim_R", "stim_L", True),
                       ("control, without MISMATCH", "test_R", "test_L", True),
                       ("fastigial photo-activation, >= 3 trials per condition (cds_lab)", "stim_R", "stim_L", "lab")]:
    xs, ys = [], []
    for r in (cds_lab if excl == "lab" else cds):
        if r["group"] != "fastigial" or (excl is True and r["session"] in MISMATCH):
            continue
        idx = np.concatenate([r["sets"][a], r["sets"][b]])
        lk = r["licked"][idx]
        ok = lk != ""
        xs.extend(r["proj"][idx[ok]][:, k].ravel()); ys.extend((lk[ok] == "R").astype(int))
    x_, y_ = np.array(xs), np.array(ys)
    fit = sm.Logit(y_, sm.add_constant(x_ / x_.std())).fit(disp=0)
    ci = fit.conf_int()[1]
    q = pd.qcut(x_, 4, labels=False)
    print(f"{nm}: trials {len(x_)}, slope {fit.params[1]:.2f} [{ci[0]:.2f}, {ci[1]:.2f}] per s.d., p = {fit.pvalues[1]:.2g}; "
          f"P(lick right) by projection quartile {[round(float(y_[q == i].mean()), 2) for i in range(4)]}")
print("sessions:", sum(r["group"] == "fastigial" for r in cds))

**Result: not reproduced.** On 181 photo-activation trials the projection still predicts the lick
direction: slope 0.54 [0.19, 0.90] per s.d., p = 0.003, and P(lick right) 0.43, 0.49, 0.47 and 0.73
by quartile, against a slope of 1.85 on control trials. Without the sessions in MISMATCH the slope is
0.38 [0.03, 0.74] (p = 0.035), and with the lab's session rule 0.30 [0.02, 0.58]
(p = 0.038, 220 trials): weaker than on control
trials, but significant in every variant.

## Summary

The outcomes, with the numbers they rest on, are in the attempt record
(`attempts/gao-2018-02.yaml`); the table here gives them in brief.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| claim | outcome | main numbers (printed above) |
|---|---|---|
| fig1g-fastigial-activation-early-sample-no-effect | reproduced | contra -0.031, ipsi +0.012; p for a drop 0.36 and 0.64 |
| fig1g-fastigial-activation-delay-errors | reproduced | 0.870 to 0.630, p < 0.0001 (6 mice, 21 sessions) |
| ed10b-fastigial-activation-errors-during-recording | reproduced | same comparison, p < 0.0001 |
| fig1h-fastigial-activation-delay-bias | reproduced | 2 mice contra-biased, 2 ipsi-biased (early delay) |
| fig1g-dentate-activation-little-effect | partially reproduced | p = 0.107 on all sessions, 0.042 without MISMATCH (3 of 8 mice) |
| fig1k-inhibition-delay-bias | reproduced | 1.3 s photo-inhibition: mice biased both ways under either laterality |
| fig1i-fastigial-activation-licking-unaffected | blocked | early licks pooled 0.083 to 0.130, p = 0.33 (BAYLORCD6 rises); bout measures need lick times |
| fig1l-inhibition-execution-unaffected | blocked | early licks pooled p = 0.13 and 0.22 (BAYLORCD8 differs); bout measures need lick times |
| fig2-alm-selective-fraction | reproduced | 856 of 1084 (0.79), same units as the lab's compile |
| fig2-preparatory-more-in-alm | partially reproduced | ALM 0.563 of all neurons; CN not deposited |
| fig2-peri-movement-equal | partially reproduced | ALM 0.642; CN not deposited |
| fig2cd-selectivity-emerges-gradually | partially reproduced | ALM holds under reading (a); CN not deposited |
| fig3d-alm-altered-by-cn-inhibition | not reproduced | 228 of 486; latency 15 to 42 ms, not 7.1 |
| fig3e-alm-mean-rate-unchanged | reproduced | 4.03 and 4.21 spikes/s, p = 0.51 |
| fig3e-alm-increase-decrease | not reproduced | 122 and 106, against 108 and 89 |
| fig3f-alm-selectivity-abolished-by-cn-inhibition | partially reproduced (not reproduced proposed) | 1.46 to 0.39 spikes/s, interval excludes zero |
| fig4-fastigial-activation-alm-latency | partially reproduced | latency 3.6 ms (bin start only); 177 modulated, not 227 |
| fig4e-fastigial-activation-abolishes-alm-selectivity | reproduced | 328 neurons; 163 of 183 lose selectivity, 6 gain; holds at matched trial counts |
| fig4f-fastigial-effect-persists | reproduced | 0.13 of neurons >= 1 s after offset, 0.009 before onset |
| fig4g-dentate-effect-recovers | partially reproduced | latency 3.30 ms by bin start (8.30 by centre) and 172 modulated match; late fraction still above baseline |
| fig4i-cd-control-separation | reproduced | 20 sessions; separated from -2.06 s and through the delay |
| fig4i-fastigial-collapses-cd | partially reproduced | fastigial -67%; dentate -14%, interval excludes zero |
| cd-captures-selectivity | partially reproduced | selectivity 66.3%; variance 35.9% to 58.9%, not 27.0 |
| ed10d-fastigial-endpoints-no-prediction | not reproduced | slope 0.54 per s.d., p = 0.003 |
| 22 other claims | blocked, data_missing | lesions, behavior-only sessions, CN recordings, VGAT, anatomy |

How the analysis differs from the paper's. Behavior is from the recording sessions, not the
behavior sessions. The data hold 34 of the 39 Fig. 2 sessions and more units than the paper
reports for these mice (1084 pyramidal neurons in the photo-activation sessions against 912; 682
against 454 in the photo-inhibition sessions); the lab's compiled analysis files hold exactly the
units here, so counts are compared with the paper only where the criterion says so. Details the
Methods leave open come from the deposited scripts: trial selection, the 1.3 s response window,
the minimum trial counts for photostimulation tests and the coding-direction neuron selection,
which keeps 20 of 34 sessions (the lab's compile 23, the paper 28 of 39). The per-neuron
photo-activation p-values of the 400 ms bins (fig4f) equal those in the lab's FR compile wherever
it has a value; the window p-values of the modulation test and the photo-inhibition p-values were
not compared with the lab's files. Four photo-activation sessions
whose laser traces disagree with their codes (MISMATCH) are kept in the main analyses and left out
in a variant; leaving them out changes the dentate behavior result (a drop at p = 0.042) and makes
both conditions of fig4i-fastigial-collapses-cd hold in the last 400 ms, and changes no other
conclusion.

What was found about the datasets. The two deposits hold the same 54 sessions with identical
spike times; DANDI lacks 26 and 268 trials in two sessions, rounds trial start times, and
describes event times as relative to the trial start when they are in session time. For the
three photo-inhibition mice CD12, CD13 and CD14 both deposits place the fastigial fiber on the
same side as the recorded ALM (both printed in the data check), against the paper's and the codes' contralateral design. The laser
traces confirm the stimulation timing in the 2017 and 2018 sessions; in four sessions they show a
1.3 s light, varying durations, no light, or delay-coded trials lit at sample onset (laser table).
The power channel of the photo-inhibition sessions reads 0.6 to 0.7 or 1.7 to 1.9 mW on average
while the light is on, which matches neither of the 1.5 and 4.5 mW the Methods state. The CRCNS
descriptor says the time series record tongue movements, but every session holds only the laser
channels (`aom_input_trace`, `laser_power`), so no lick times are deposited.

In [ ]:
print(f"total runtime {(time.time() - started) / 60:.1f} min, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")